In [1]:
import json
from pathlib import Path
import httpx
from typing import List, Dict
import threading
import time
from http.server import HTTPServer, BaseHTTPRequestHandler
from urllib.parse import urlparse

# 服务器配置
server11 = "10.192.72.11:6300"
server12 = "10.192.72.12:6300"
localhost = "127.0.0.1:6300"

# 使用本地服务器
base_url = f"http://{localhost}"
callback_url = f"{base_url}/api/content_generate/callback"

# 测试开关：若为 True，则回调服务器在收到 start 事件后立刻返回 interrupt=true
TEST_INTERRUPT = False

print(f"回调接口地址: {callback_url}")
print(f"TEST_INTERRUPT={TEST_INTERRUPT}")
print("done")


回调接口地址: http://127.0.0.1:6300/api/content_generate/callback
TEST_INTERRUPT=False
done


In [2]:
import json
from pathlib import Path
import random

content_file_path = Path("../../storage/merged_converted_output_poc1226.json")
# content_file_path = Path("../storage/merged_converted_output_xwkd1217.json")
with content_file_path.open("r", encoding="utf-8") as f:
    content_list = json.load(f)

question_tasks_file_path = Path("../../storage/question_tasks_poc1226.json")
# question_tasks_file_path = Path("../storage/question_tasks_xwkd1217.json")
with question_tasks_file_path.open("r", encoding="utf-8") as f:
    question_tasks_from_file_full : List[Dict] = json.load(f)
    # 随机取5个问题
    # question_tasks_from_file = random.sample(question_tasks_from_file_full, 5)
    # 暂时不考虑table类型问题
    question_tasks_from_file_table = [q for q in question_tasks_from_file_full if q["type"] == "table"]
    question_tasks_from_file_nontable = [q for q in question_tasks_from_file_full if q["type"] != "table"]

question_tasks = [
    # {
    #     "id": "#1",
    #     "description": "申请人的基础设施建设业务有哪些？基础设施板块中PPP项目的数量、投资与回款情况如何？回报机制以何种方式为主？是否存在回款周期长和运营不确定性风险？",
    #     "prompt": "申请人的名称是什么？请翻译为英文回答；",
    #     "type": "table",
    #     "html": "",
    # }, 
    {
        "id": "q_1",
        "description": "前五大下游企业明细表，提取：序号、客户名称、是否存在关联关系、是否国企、采购金额（万元）",
        "prompt": "提取：序号、客户名称、是否存在关联关系、是否国企、采购金额（万元）                  示例：[{\"序号\": \"1\",\"客户名称\": \"客户一\",\"是否存在关联关系\": \"\",\"是否国企\": \"\",\"采购金额\": \"\"}]",
        "type": "text-json",
        "html": "",
        "human": "True"
    },
    # {
    #     "id": "#2",
    #     "description": "请填写公司的注册资本。请简要回答",
    #     "prompt": "",
    #     "type": "text",
    #     "html": "",
    #     "human": "True"
    # },
    #     {
    #     "id": "t_2",
    #     "description": "分行初审意见及身份证号",
    #     "prompt": "",
    #     "type": "table",
    #     "html": "",
    #     "data_sources": [
    #         {
    #             "type": "chunk_index_retrieval",
    #             "query": "分行初审意见",
    #             "top_k": 30,
    #             "return_child_chunk": True,
    #             "rerank_enabled": False,
    #             "hybrid_enabled": True,
    #             "search_table": False,
    #         },
    #         {
    #             "type": "extract",
    #             "data": [
    #                 {
    #                 "tagName": "身份证",
    #                 "files": [
    #                     {
    #                     "fileId": "..",
    #                     "objectSource": [
    #                         {
    #                             "element_id": "1",
    #                             "value": {
    #                                 "姓名": "张三"
    #                             }
    #                         },
    #                         {
    #                             "element_id": "6",
    #                             "value": {
    #                                 "身份证号": "110111100020008765"
    #                             }
    #                         }
    #                     ]
    #                     }
    #                 ]
    #                 },
    #             ]
    #         }
    #     ],
    #     "tools":[
    #         {
    #             "type": "coding_agent",
    #             "enable": True
    #         }
    #     ]
    # },
    {
        "id": "q_2e",
        "description": "Branch Preliminary Review Opinion and ID Number",
        "prompt": "",
        "type": "text",
        "html": "",
        "data_sources": [
            {
                "type": "chunk_index_retrieval",
                "query": "分行初审意见",
                "top_k": 30,
                "return_child_chunk": True,
                "rerank_enabled": False,
                "hybrid_enabled": True,
                "search_table": False,
            },
            {
                "type": "extract",
                "data": [
                    {
                    "tagName": "ID Card",
                    "files": [
                        {
                        "fileId": "...略",
                        "objectSource": [
                            {
                                "element_id": "1",
                                "value": {
                                    "Name": "Grizzm"
                                }
                            },
                            {
                                "element_id": "6",
                                "value": {
                                    "ID Number": "110111100020008765"
                                }
                            }
                        ]
                        }
                    ]
                    },
                ]
            }
        ],
        "tools":[
            {
                "type": "coding_agent",
                "enable": True
            }
        ]
    },
    # {
    # "id": "q_28",
    # "description": "生成一份 "近年来公司新项目承揽情况" 的统计表格：首先明确表格需包含的表头为 "项目"、材料中所给的三年一期；其次确定需要填充的项目对应的各年份 / 期间数据，包括 "新签合同金额"（单位为亿元）、"政国企类占比"（单位为 %）、"5 亿元及以上合同金额占比"（单位为 %）这三类项目在三年一期对应的数值，请按此表头和项目整理成规范表格，并根据表格的设置找到对应数据并填充",
    # "type": "table",
    # "html": "<table><tr><td>项目</td><td>2022年</td><td>2023年</td><td>2024年</td><td>2025.1-3</td></tr></table>",
    # "contents": ""
    # },
]

# 若需测试「按题加载额外文件」：可为某题添加 "knowledge": ["file_id1", "file_id2"]，详见下方「Knowledge 功能测试」小节
print(f"问题任务数量: {len(question_tasks)}")


问题任务数量: 2


In [ ]:
from datetime import datetime
import uuid

def get_timestamp():
    """获取当前时间戳字符串"""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]


def print_event(event_type: str, data: any):
    """输出事件 - 改进版本，增加错误处理和日志，包含时间戳"""
    timestamp = get_timestamp()
    try:
        if event_type == "message":
            print(f"[{timestamp}] [消息事件] {data}")
        elif event_type == "heartbeat":
            print(f"[{timestamp}] [心跳事件] {data}")
        elif event_type == "complete":
            print(f"[{timestamp}] [完成事件] {data}")
        elif event_type == "error":
            print(f"[{timestamp}] [错误事件] {data}")
        elif event_type == "start":
            print(f"[{timestamp}] [开始事件] {data}")
        else:
            print(f"[{timestamp}] [未知事件] 类型: {event_type}, 数据: {data}")
    except Exception as e:
        print(f"[{timestamp}] [错误] 处理事件时出错: {e}, 事件类型: {event_type}, 数据: {data}")


# 回调接收服务器
def create_callback_handler(callback_queue):
    """创建回调处理器类"""
    class CallbackHandler(BaseHTTPRequestHandler):
        """回调请求处理器"""
        
        def do_POST(self):
            """处理POST回调请求"""
            try:
                content_length = int(self.headers.get('Content-Length', 0))
                post_data = self.rfile.read(content_length)
                callback_data = json.loads(post_data.decode('utf-8'))
                
                # 解析回调数据
                event = callback_data.get('event')
                data_str = callback_data.get('data', '{}')
                task_id = callback_data.get('taskId')
                
                # 解析data字段（JSON字符串）
                try:
                    data = json.loads(data_str) if isinstance(data_str, str) else data_str
                except:
                    data = data_str
                
                # 将回调数据放入队列
                callback_queue.put({
                    'event': event,
                    'data': data,
                    'taskId': task_id
                })
                
                # 返回响应（用于服务端决定是否中断）
                # 格式：{"data": {"taskId": "...", "interrupt": true/false}, ...}
                interrupt = bool(TEST_INTERRUPT and event == 'start')
                resp = {
                    'code': '0',
                    'total': 0,
                    'message': None,
                    'level': 'info',
                    'data': {
                        'taskId': task_id,
                        'interrupt': interrupt
                    },
                    'i18nData': None,
                    'extData': None
                }
                if interrupt:
                    print(f"[{get_timestamp()}] [TEST_INTERRUPT] 收到start事件，返回中断信号: {resp}")
                
                self.send_response(200)
                self.send_header('Content-Type', 'application/json')
                self.end_headers()
                self.wfile.write(json.dumps(resp).encode('utf-8'))
                
            except Exception as e:
                print(f"[{get_timestamp()}] [回调服务器错误] {e}")
                self.send_response(500)
                self.end_headers()
        
        def log_message(self, format, *args):
            """禁用默认日志输出"""
            pass
    
    return CallbackHandler


def run_callback_test(content_list, question_tasks, callback_port=8888, timeout_seconds=1800, answer_language=None):
    """
    运行回调接口测试
    
    Args:
        content_list: 文档内容列表
        question_tasks: 问题任务列表
        callback_port: 回调接收服务器端口
        timeout_seconds: 超时时间（秒）
        answer_language: 回答语言 locale，如 "en_US" / "zh_CN"，不传则使用服务端默认（zh_CN）
    """
    import queue
    
    # 创建回调队列
    callback_queue = queue.Queue()
    
    # 创建回调接收服务器
    callback_url_local = f"http://127.0.0.1:{callback_port}/callback"
    # callback_url_local = "http://10.192.72.11:22223/api/aitask/modelCallBack"
    
    CallbackHandler = create_callback_handler(callback_queue)
    server = HTTPServer(('127.0.0.1', callback_port), CallbackHandler)
    
    # 在后台线程中启动服务器
    server_thread = threading.Thread(target=server.serve_forever, daemon=True)
    server_thread.start()
    
    print(f"[{get_timestamp()}] [开始] 回调接收服务器已启动，监听端口: {callback_port}")
    print(f"[{get_timestamp()}] [回调URL] {callback_url_local}\n")
    
    # 生成任务ID
    task_id = f"TEST_{uuid.uuid4().hex[:8]}"
    
    # 准备请求体
    payload = {
        "contentList": content_list,
        "questionTasks": question_tasks,
        "taskId": task_id,
        "callbackUrl": callback_url_local  # 使用debug模式，指定回调URL TODO
    }
    # 回答语言：通过标准 Header Accept-Language 传递（如 en_US 测试英文回答）
    headers = {"Accept-Language": answer_language} if answer_language else {}
    
    results = []
    final_result = None
    start_time = time.time()
    
    try:
        # 发送POST请求
        with httpx.Client(timeout=30.0) as client:
            response = client.post(callback_url, json=payload, headers=headers)
            
            if response.status_code != 200:
                print(f"[{get_timestamp()}] [错误] HTTP状态码: {response.status_code}")
                try:
                    error_text = response.text
                    print(f"响应内容: {error_text}")
                except:
                    pass
                return None
            
            response_data = response.json()
            print(f"[{get_timestamp()}] [接口响应] {response_data}\n")
        
        # 等待接收回调
        print(f"[{get_timestamp()}] [等待回调] 任务ID: {task_id}\n")
        
        while True:
            # 检查超时
            elapsed = time.time() - start_time
            if elapsed > timeout_seconds:
                print(f"\n[{get_timestamp()}] [错误] 等待回调超时 ({timeout_seconds}秒)")
                break
            
            try:
                # 从队列获取回调数据（非阻塞）
                callback_item = callback_queue.get(timeout=1.0)
                
                event = callback_item['event']
                data = callback_item['data']
                
                # 打印事件
                print_event(event, data)
                
                # 保存消息事件
                if event == "message":
                    results.append(data)
                elif event == "complete":
                    final_result = data
                    print(f"\n[{get_timestamp()}] [完成] 共接收到 {len(results)} 个message事件")
                    break
                elif event == "error":
                    final_result = data
                    print(f"\n[{get_timestamp()}] [错误] 任务失败")
                    break
                    
            except queue.Empty:
                # 队列为空，继续等待
                continue
            except Exception as e:
                print(f"[{get_timestamp()}] [错误] 处理回调时出错: {e}")
                import traceback
                traceback.print_exc()
                continue
        
        if final_result:
            print(f"\n[{get_timestamp()}] [最终结果]:")
            print(json.dumps(final_result, ensure_ascii=False, indent=2))
        
        return {
            "results": results,
            "final_result": final_result,
            "task_id": task_id
        }
        
    except httpx.TimeoutException:
        print(f"\n[{get_timestamp()}] [错误] 请求超时")
        return None
    except Exception as e:
        print(f"\n[{get_timestamp()}] [错误] 执行任务时出错: {e}")
        import traceback
        traceback.print_exc()
        return None
    finally:
        # 关闭服务器
        server.shutdown()
        print(f"[{get_timestamp()}] [结束] 回调接收服务器已关闭")


# 运行测试（answer_language="en_US" 时答案以英文返回，不传或 "zh_CN" 为中文）
result = run_callback_test(content_list, question_tasks, callback_port=8888, answer_language="en_US")


[2026-04-16 14:38:04.566] [开始] 回调接收服务器已启动，监听端口: 8888
[2026-04-16 14:38:04.566] [回调URL] http://127.0.0.1:8888/callback

[2026-04-16 14:38:08.562] [接口响应] {'success': True, 'message': 'The task has been submitted and is being processed in the background.', 'taskId': 'TEST_0bf44c48'}

[2026-04-16 14:38:08.563] [等待回调] 任务ID: TEST_0bf44c48

[2026-04-16 14:38:09.041] [开始事件] {'msg': 'Processing started'}
[2026-04-16 14:39:45.267] [消息事件] {'questionId': 'q_2e', 'questionText': 'Branch Preliminary Review Opinion and ID Number', 'questionType': 'text', 'answer': {'isAnswerable': False, 'answerText': "Insufficient information to determine the Branch Preliminary Review Opinion; although the ID Number is provided as '110111100020008765' , there is no context regarding any preliminary review opinion from a branch.", 'citations': [], 'thinkingSteps': {'chunkRetrieval': None, 'summaryRetrieval': None, 'documentRetrieval': None, 'answerAgentSteps': [{'type': 'planner', 'text': '问题要求的是‘Branch Preliminary R

## Knowledge 功能测试

测试「从外部接口读取问题的额外文件内容」：问题可配置 `knowledge`（file_id 列表），后端请求 `CONTENT_GENERATE_CALLBACK_URL/api/aidoc/info` 获取 `data.docParseContent`，仅作为该题的附加上下文。

**前置条件**：content_generate 服务启动时设置 `CONTENT_GENERATE_CALLBACK_URL=http://127.0.0.1:8889`（与下方 Mock 端口一致）。

In [ ]:
# # ========== Knowledge 功能测试：从外部接口读取问题的额外文件内容 ==========
# # 说明：每个问题可配置 knowledge 字段（file_id 列表），后端会请求
# #   CONTENT_GENERATE_CALLBACK_URL/api/aidoc/info?fileId=xxx 获取 data.docParseContent，
# #   并仅将该内容作为该问题的附加检索/生成上下文。
# # 测试前请确保 content_generate 服务启动时已设置环境变量：
# #   CONTENT_GENERATE_CALLBACK_URL=http://127.0.0.1:8889
# # （与下方 Mock 服务端口一致；若使用真实 aidoc 服务则改为其 base URL。）

# from http.server import HTTPServer, BaseHTTPRequestHandler
# from urllib.parse import urlparse, parse_qs

# # from datetime import datetime
# # import threading
# # def get_timestamp():
# #     """获取当前时间戳字符串"""
# #     return datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]

# # Mock 返回的 docParseContent：与 content_list 中单条 content 格式一致（Item 列表）
# MOCK_DOC_PARSE_CONTENT = [
#     {
#         "page": 1,
#         "type": "text",
#         "bBox": {"l": 0, "t": 0, "r": 100, "b": 20, "coord_origin": "TOPLEFT"},
#         "value": "【Mock 附加文件】本段内容来自 knowledge 接口，用于验证按题加载额外文件功能。信息：本段 Mock 附加文件说明了附加文件内容能够正常使用。",
#         "md": "【Mock 附加文件】本段内容来自 knowledge 接口，用于验证按题加载额外文件功能。信息：本段 Mock 附加文件说明了附加文件内容能够正常使用。",
#         "file_id": "7b4778392621dd0a5420dd639362f4e5",
#         "element_id": 1,
#     }
# ]


# class MockAidocInfoHandler(BaseHTTPRequestHandler):
#     """Mock GET/POST /api/aidoc/info，返回 data.docParseContent。"""

#     def _handle_info(self):
#         parsed = urlparse(self.path)
#         if parsed.path != "/api/aidoc/info":
#             self.send_response(404)
#             self.end_headers()
#             return
#         params = parse_qs(parsed.query)
#         file_ids = params.get("fileId", [])
#         file_id = file_ids[0] if file_ids else "unknown"
#         # 可据 file_id 返回不同内容，此处统一返回同一 mock 内容
#         body = {
#             "data": {
#                 "docParseContent": MOCK_DOC_PARSE_CONTENT,
#             }
#         }
#         self.send_response(200)
#         self.send_header("Content-Type", "application/json; charset=utf-8")
#         self.end_headers()
#         self.wfile.write(json.dumps(body, ensure_ascii=False).encode("utf-8"))

#     def do_GET(self):
#         self._handle_info()

#     def do_POST(self):
#         # 当前实现中，后端通过 POST + query 参数调用 /api/aidoc/info
#         # 这里直接复用 GET 逻辑，仅基于 URL 查询参数解析 fileId
#         self._handle_info()

#     def log_message(self, format, *args):
#         pass


# # 启动 Mock 服务（与主测试的回调服务器端口错开）
# MOCK_AIDOC_PORT = 8889
# mock_aidoc_server = HTTPServer(("127.0.0.1", MOCK_AIDOC_PORT), MockAidocInfoHandler)
# mock_thread = threading.Thread(target=mock_aidoc_server.serve_forever, daemon=True)
# mock_thread.start()
# print(f"[{get_timestamp()}] [Knowledge Mock] /api/aidoc/info 已启动: http://127.0.0.1:{MOCK_AIDOC_PORT}/api/aidoc/info")
# print(f"  请确保 content_generate 服务启动时设置: CONTENT_GENERATE_CALLBACK_URL=http://127.0.0.1:{MOCK_AIDOC_PORT}")

In [ ]:
# # 仅含 1 个带 knowledge 的问题，用于验证按题加载额外文件
# question_tasks_knowledge = [
#     {
#         "id": "q_knowledge_test",
#         "description": "请根据附加文件内容回答：本段 Mock 附加文件说明了什么？",
#         "prompt": "",
#         "type": "text",
#         "html": "",
#         "knowledge": ["7b4778392621dd0a5420dd639362f4e5"],
#         "tools": [{"type": "coding_agent", "enable": False}],
#     }
# ]

# # 可选：先验证 Mock 接口返回格式正确
# try:
#     r = httpx.post(f"http://127.0.0.1:{MOCK_AIDOC_PORT}/api/aidoc/info", params={"fileId": "7b4778392621dd0a5420dd639362f4e5"}, timeout=5.0)
#     r.raise_for_status()
#     data = r.json()
#     content = data.get("data", {}).get("docParseContent")
#     assert isinstance(content, list) and len(content) > 0, "Mock 应返回 data.docParseContent 非空列表"
#     print(f"[{get_timestamp()}] [验证] Mock /api/aidoc/info 返回 docParseContent 条数: {len(content)}")
# except Exception as e:
#     print(f"[{get_timestamp()}] [警告] Mock 验证失败（请先运行上一 cell 启动 Mock）: {e}")

# # 运行 Knowledge 测试（会启动回调服务器 8888，与 Mock 8889 独立）
# result_knowledge = run_callback_test(content_list, question_tasks_knowledge, callback_port=8888)
# if result_knowledge and result_knowledge.get("results"):
#     for msg in result_knowledge["results"]:
#         qid = msg.get("questionId")
#         ans = msg.get("answer", {})
#         text = ans.get("answerText", "")
#         print(f"\n[{get_timestamp()}] [Knowledge 测试] 问题 {qid} 答案摘要: {text[:200]}...")